In [1]:
!powercfg /setactive 8c5e7fda-e8bf-4a96-9a85-a6e23a8c635c

In [2]:
import pickle

with open("measurements_by_frame.pkl", "rb") as f:
    measurements_by_frame = pickle.load(f)

with open("timestamps.pkl", "rb") as f:
    timestamps = pickle.load(f)

In [3]:
# radar measurement in form [range, azimuth, doppler]
measurements_by_frame[720]

array([[11.14649682,  0.10471976, -2.21586147],
       [11.46496815,  0.03490659, -1.70450883],
       [12.10191083,  0.01745329, -2.55676324],
       [12.42038217,  0.01745329, -2.72721412]])

In [4]:
# relative elapsed time stamp in seconds
timestamps[720]

np.float64(36.156)

In [5]:
from customed_dbscan import cluster_measurements


clustered_measurements_by_frame = []
for measurements in measurements_by_frame:
    clustered_measurements = cluster_measurements(
    measurements,
    sigma_range=2.0,  # range uncertainty 
    sigma_azimuth_deg=1.0,
    sigma_doppler=2.0,
    street_angle_deg=0.0,   # the direction of clustering
    x_scale=2.0,   
    y_scale=2.0,
    dbscan_eps=1.0,
    dbscan_min_samples=2
)
    
    clustered_measurements_by_frame.append(
        cluster_measurements(measurements)
    )


In [6]:
# z: the cluster mean
# R: the cluster measurement noise, will be used in the tracker (notice R now is meas dependent!)
clustered_measurements_by_frame[720]

[{'z': array([11.78343949,  0.04362525, -2.30108691]),
  'R': array([[ 1.08451999e+00, -5.09517563e-03, -4.97600772e-02],
         [-5.09517563e-03,  5.07695721e-04,  1.48746456e-03],
         [-4.97600772e-02,  1.48746456e-03,  1.05084363e+00]])}]

In [11]:
import numpy as np
from config import TrackerConfig
from radar_jpda_tracker import RadarJPDAEKF


# ============================================================
# TRACKER CONFIGURATION
# ============================================================

config = TrackerConfig(

    # --------------------------------------------------------
    # MEASUREMENT NOISE
    # --------------------------------------------------------

    sigma_range=0.3,

    sigma_angle=np.deg2rad(1.0),

    sigma_doppler=0.17,

    # --------------------------------------------------------
    # ACCELERATION NOISE
    # --------------------------------------------------------

    sigma_ax=1.0,

    sigma_ay=1.0,

    # --------------------------------------------------------
    # JPDA
    # --------------------------------------------------------

    pd=0.90,

    pg=0.99,

    clutter_density=1e-4,

    # --------------------------------------------------------
    # ASSOCIATION
    # --------------------------------------------------------

    association_probability_threshold=0.70,  # only to count a hit or not, it is not considered in calculations

    # --------------------------------------------------------
    # TRACK MANAGEMENT
    # --------------------------------------------------------

    max_missed=6,

    confirmation_hits=4,

    min_range=0.2,

    max_tracks=40,
    
    # TO DECIDE WETHER A NEW TRACK SHOUD BE OPENED BASED ON MEAS ASSOCIATION PROBABILITY
    # ZERO MEANS ANY MEAS FALLS INSIDE THE GATE WILL BE ASSOCIATED EVEN WITH A LOW PROBABILITY
    measurement_probability = 0.0,   

    # --------------------------------------------------------
    # MERGE CLOSED TRACKS
    #
    # ONLY EUCLIDEAN POSITION DISTANCE
    # --------------------------------------------------------

    merge_distance_threshold=0.0,

    merge_confirmed_only=False,

    # --------------------------------------------------------
    # TIMING
    # --------------------------------------------------------

    default_dt=0.1,

    # --------------------------------------------------------
    # INITIALIZATION
    # --------------------------------------------------------

    # --------------------------------------------------------
    # Minimum covariance eigenvalue to prevent singular / ill-conditioned matrices
    # --------------------------------------------------------

    covariance_eigenvalue_floor=1e-10,

    # --------------------------------------------------------
    # DEBUG
    # --------------------------------------------------------

    debug=False)

# ============================================================
# CREATE TRACKER
# ============================================================

tracker = RadarJPDAEKF(config)

In [12]:
import pandas as pd

# --------------------------------------------------------
# Run tracker + store all tracker information
# --------------------------------------------------------

tracker_records = []

for frame_index, (measurements, timestamp) in enumerate(
    zip(clustered_measurements_by_frame, timestamps)
):
    states = tracker.step(
        measurements,
        timestamp
    )

    #print(f"\n========================================== FRAME {frame_index} ====================================================")

    for state in states:
        x = state["x"]
        if state["confirmed"] == True:

            # Store everything we know about this track at this frame
            record = {
                "frame_index": frame_index,
                "timestamp": timestamp,

                "track_id": state["id"],
                "P": state["P"],
                "age": state["age"],

                "px": x[0],
                "py": x[1],
                "vx": x[2],
                "vy": x[3],

                "hits": state["hits"],
                "missed": state["missed"],
                "confirmed": state["confirmed"],
            }

            tracker_records.append(record)


# --------------------------------------------------------
# Create Pandas DataFrame
# --------------------------------------------------------

tracker_df = pd.DataFrame(tracker_records)


In [14]:
pd.set_option("display.max_rows", None)

tracker_df.iloc[0:3]

,frame_index,timestamp,track_id,P,age,px,py,vx,vy,hits,missed,confirmed
0,356,17.905,0,"[[0.5888861529788229, -0.09688223927779936, 0....",7,37.433895,-10.017700,-3.843371,3.973029,4,0,True
1,357,17.955,0,"[[0.516844750935161, -0.07996298292895063, 0.1...",8,37.223213,-10.201364,-4.354378,2.155911,5,0,True
2,358,18.005,0,"[[0.4257354423725727, -0.05264842814987513, 0....",9,36.987662,-10.174339,-4.485170,1.812154,6,0,True
